# Tata Steel — Roller-Table Motor Predictive & Prescriptive Maintenance

**Motor ID:** ROT-MILL-05 | **Process:** TSCR Roller Table | **Sampling Rate:** 1 second

### Pipeline:
1. Full exploratory analysis of all 10 sensor columns
2. Statistical threshold derivation and feature-selection justification
3. ML model training, cross-validation, and benchmarking
4. Health-index scoring and sensor-aware prescriptive maintenance

**Dataset:** `tata_steel_rot_motor_proxy.csv` — 10,000 rows, 1-second intervals, simulated IoT data for a 415 V / 30 kW roller-table motor.

## Part 1 — Exploratory Data Analysis

We load all columns first, understand the data structure, then use correlation and distribution analysis to decide which features matter for modelling.

### 1.1 Data Loading & Overview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("tata_steel_rot_motor_proxy.csv")

print(f"Shape : {df.shape}")
print(f"Columns: {list(df.columns)}")
print("\n--- Data Types & Nulls ---")
df.info()
print("\n--- Descriptive Statistics (all numeric) ---")
df.describe().round(2)

### 1.2 Correlation Heatmap — All Numeric Sensors

This tells us which sensors move together (shared cause) and which are independent noise. We use this to justify feature selection for the ML model.

In [ ]:
numeric_cols = ["Current_Amp", "Voltage_V", "Motor_RPM", "Vibration_mm_s",
                "Winding_Temp_C", "Bearing_Temp_C", "Coolant_Pressure_Bar", "Ambient_Humidity_Pct"]

plt.figure(figsize=(9, 7))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, linewidths=0.5, square=True
)
plt.title("Sensor Correlation Matrix (all 8 numeric columns)", fontsize=13, pad=14)
plt.tight_layout()
plt.show()

**Observations:**
- **Current, RPM, Vibration, and both Temperatures** are correlated — they all respond to the cyclic slab-loading pattern.
- **Voltage, Coolant Pressure, and Humidity** show near-zero correlation with everything else — independent environmental noise.

**Feature selection decision:** `Coolant_Pressure_Bar` and `Ambient_Humidity_Pct` are excluded from the ML feature set. `Voltage` is kept because supply fluctuations can directly affect current draw.

### 1.3 Data Visualization

The motor cycles between idling (45 A) and loaded (85 A) as steel slabs pass.

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(12, 4))

axes.plot(df["Current_Amp"].iloc[:500], color="#5B9BD5", linewidth=0.8)
axes.axhline(60, color="gray", linestyle=":", alpha=0.6, label="~60 A midpoint")
axes.set_xlabel("Time Index (seconds)")
axes.set_ylabel("Current (A)")
axes.set_title("Current Over Time — Cyclic Load Switching (first 500 s)")
axes.legend()

plt.tight_layout()
plt.show()

### 1.4 Temperature as a Lagging Indicator

Temperatures change slowly — they smooth towards a target rather than jumping instantly with load. Winding temp responds faster than bearing temp. This makes them good for tracking steady-state health, but not for detecting sudden faults — that role belongs to vibration.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
t = 800

axes[0].plot(df["Current_Amp"].iloc[:t],    color="#5B9BD5", linewidth=0.7, label="Current (A)")
axes[0].plot(df["Winding_Temp_C"].iloc[:t], color="#ED7D31", linewidth=1.2, label="Winding Temp (C)")
axes[0].set_xlabel("Time Index (seconds)")
axes[0].set_ylabel("Value")
axes[0].set_title("Current vs Winding Temperature — Lagging Response")
axes[0].legend(loc="upper right")

axes[1].plot(df["Current_Amp"].iloc[:t],     color="#5B9BD5", linewidth=0.7, label="Current (A)")
axes[1].plot(df["Bearing_Temp_C"].iloc[:t],  color="#70AD47", linewidth=1.2, label="Bearing Temp (C)")
axes[1].set_xlabel("Time Index (seconds)")
axes[1].set_ylabel("Value")
axes[1].set_title("Current vs Bearing Temperature — Even Slower Response")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

### 1.5 Rule-Based Anomaly Detection — Vibration Spikes

Instead of a hardcoded limit, all three thresholds below are **statistically derived** from the data:

| Threshold | Method | Principle |
|---|---|---|
| **Vibration** | `loaded_mean + 3 × loaded_std` | State-separated 3-sigma (idle rows excluded to avoid noise dilution) |
| **Bearing Temp** | `mean + 3 × std` | Global 3-sigma process-control rule (covers 99.73% of normal readings) |
| **Winding Temp** | `mean + 3 × std` | Same global 3-sigma rule |

All three values are computed in the code cell below and printed before being used.

In [ ]:
# -- Vibration Threshold: state-separated 3-sigma (process-control principle) --
# Principle: separate idle vs loaded states so idle noise does not dilute the loaded-state sigma
idle_mask   = df["Current_Amp"] < 60
loaded_mask = df["Current_Amp"] >= 60

idle_vib_mean   = df.loc[idle_mask,   "Vibration_mm_s"].mean()
idle_vib_std    = df.loc[idle_mask,   "Vibration_mm_s"].std()
loaded_vib_mean = df.loc[loaded_mask, "Vibration_mm_s"].mean()
loaded_vib_std  = df.loc[loaded_mask, "Vibration_mm_s"].std()

# Threshold based on loaded state only (state-separated 3-sigma)
vibration_threshold = loaded_vib_mean + 3 * loaded_vib_std
print(f"Vibration threshold (loaded_mean + 3*loaded_std): {vibration_threshold:.4f} mm/s")

# -- Temperature Thresholds: global 3-sigma (process-control principle) --
bearing_temp_threshold = df["Bearing_Temp_C"].mean() + 3 * df["Bearing_Temp_C"].std()
winding_temp_threshold = df["Winding_Temp_C"].mean() + 3 * df["Winding_Temp_C"].std()
print(f"Bearing Temp threshold (mean + 3*std): {bearing_temp_threshold:.4f} C")
print(f"Winding Temp threshold (mean + 3*std): {winding_temp_threshold:.4f} C")

# -- Detect vibration anomalies using the derived threshold --
df["Anomaly"] = (df["Vibration_mm_s"] > vibration_threshold).astype(int)
anomalies = df[df["Anomaly"] == 1]

print("\nNumber of vibration anomalies detected:", len(anomalies))
print(anomalies[["Timestamp", "Vibration_mm_s", "Anomaly"]].head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

# Full timeline
axes[0].plot(df["Vibration_mm_s"], color="#5B9BD5", linewidth=0.6, label="Vibration (mm/s)")
axes[0].scatter(anomalies.index, anomalies["Vibration_mm_s"],
                color="red", s=20, zorder=5, label="Anomaly (spike)")
axes[0].axhline(vibration_threshold, color="orange", linestyle="--",
                label=f"Threshold ({vibration_threshold:.2f})")
axes[0].set_title("Vibration Monitoring — Full Timeline")
axes[0].set_ylabel("Vibration (mm/s)")
axes[0].legend()

# Zoomed
z = 500
zoom_anom = anomalies[anomalies.index < z]
axes[1].plot(df["Vibration_mm_s"].iloc[:z], color="#5B9BD5", linewidth=0.9, label="Vibration (mm/s)")
axes[1].scatter(zoom_anom.index, zoom_anom["Vibration_mm_s"],
                color="red", s=45, zorder=5, label="Anomaly (spike)")
axes[1].axhline(vibration_threshold, color="orange", linestyle="--",
                label=f"Threshold ({vibration_threshold:.2f})")
axes[1].set_title(f"Vibration Monitoring — Zoomed (first {z} s)")
axes[1].set_xlabel("Time Index (seconds)")
axes[1].set_ylabel("Vibration (mm/s)")
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Part 2 — Machine Learning Model

We train an unsupervised anomaly-detection model (One-Class SVM) to learn normal equipment behavior from sensor patterns and flag out-of-distribution operating states as potential anomalies.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Common feature set used for both regression benchmarks and One-Class SVM
features = ["Motor_RPM", "Vibration_mm_s", "Winding_Temp_C", "Bearing_Temp_C", "Voltage_V"]
X = df[features].copy()
y = df["Current_Amp"].copy()

# Split used only for supervised benchmark models
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Feature matrix shape:", X.shape)
print("Features used:", features)
print(f"Train/Test rows: {len(X_train)} / {len(X_test)}")
display(X.head())

### 2.1 Model Development Setup

We first prepare the common modelling setup (features, train/test view, and anomaly outputs) so all candidate approaches can be compared on a shared basis.

At this stage, no final model is selected yet.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import numpy as np

# Features used for unsupervised anomaly detection
ocsvm_features = ["Motor_RPM", "Vibration_mm_s", "Winding_Temp_C", "Bearing_Temp_C", "Voltage_V"]

# === BUILD CLEAN TRAINING DATA: BOTH IDLE & LOADED STATES ===
# Define clean idle state (exclude extreme lows)
idle_mask = df["Current_Amp"] < 60
idle_mean_current = df.loc[idle_mask, "Current_Amp"].mean()
idle_std_current = df.loc[idle_mask, "Current_Amp"].std()
clean_idle_mask = (
    (df["Current_Amp"] >= idle_mean_current - idle_std_current) &
    (df["Current_Amp"] < 60)
)

# Define clean loaded state (exclude extreme highs)
loaded_mask = df["Current_Amp"] >= 60
loaded_mean_current = df.loc[loaded_mask, "Current_Amp"].mean()
loaded_std_current = df.loc[loaded_mask, "Current_Amp"].std()
clean_loaded_mask = (
    (df["Current_Amp"] >= 60) &
    (df["Current_Amp"] <= loaded_mean_current + loaded_std_current)
)

# Combine: training data is both clean idle + clean loaded states
clean_normal_mask = clean_idle_mask | clean_loaded_mask

X_ocsvm_all = df[ocsvm_features]
X_ocsvm_train = df.loc[clean_normal_mask, ocsvm_features]

# === TRAIN/TEST INFO ===
print("="*70)
print("FEATURE ENGINEERING & DATA SPLIT")
print("="*70)
print(f"Benchmark (regression models) — Train/Test split: 80% / 20% = {len(X_train)} / {len(X_test)} rows")
print()
print("="*70)
print("ONE-CLASS SVM UNSUPERVISED TRAINING")
print("="*70)
idle_training_rows = clean_idle_mask.sum()
loaded_training_rows = clean_loaded_mask.sum()
print(f"One-Class SVM training subset: {len(X_ocsvm_train)} ({100*len(X_ocsvm_train)/len(df):.2f}% of dataset)")
print(f"  — Clean idle rows: {idle_training_rows} ({100*idle_training_rows/len(X_ocsvm_train):.1f}% of training)")
print(f"  — Clean loaded rows: {loaded_training_rows} ({100*loaded_training_rows/len(X_ocsvm_train):.1f}% of training)")
print()

# Scale inputs (required for distance-based kernel behavior)
ocsvm_scaler = StandardScaler()
X_ocsvm_train_scaled = ocsvm_scaler.fit_transform(X_ocsvm_train)
X_ocsvm_all_scaled = ocsvm_scaler.transform(X_ocsvm_all)

# Train and infer anomalies
ocsvm_model = OneClassSVM(kernel="rbf", nu=0.01, gamma="scale")  # nu=0.01 → ~1% expected anomaly rate
ocsvm_model.fit(X_ocsvm_train_scaled)

# sklearn convention: -1 anomaly, 1 normal
df["OCSVM_Anomaly"] = ocsvm_model.predict(X_ocsvm_all_scaled)

# Higher score = more anomalous (invert signed distance)
df["OCSVM_Score"] = -ocsvm_model.decision_function(X_ocsvm_all_scaled)

ocsvm_anomalies = df[df["OCSVM_Anomaly"] == -1]
print(f"Anomalies detected: {len(ocsvm_anomalies)} ({100*len(ocsvm_anomalies)/len(df):.2f}% of all rows)")
print(f"  (Model learned both idle & loaded patterns; flags only true deviations)")
print()

### 2.2 Benchmark Comparison

We compare candidate approaches using anomaly-detection metrics only:

- **Anomaly Rate (%)**
- **Spike Coverage (%)**
- **Precision Proxy (%)**
- **F1 Proxy**

MAE, RMSE, and R2 are intentionally excluded because this section is focused on anomaly-detection quality.

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Candidate models compared only on anomaly-detection behavior
reg_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42),
    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="reg:squarederror",
        n_jobs=-1
    )
}

benchmark_rows = []

# Rule-based spike proxy on test split, used as anomaly reference
spike_proxy_test = (df.loc[X_test.index, "Vibration_mm_s"] > vibration_threshold).astype(int)

for model_name, model in reg_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    # Build anomaly flags from large regression residuals (95th percentile of train residuals)
    train_pred = model.predict(X_train)
    train_resid = (y_train - train_pred).abs()
    resid_threshold = train_resid.quantile(0.95)
    test_resid = (y_test - pred).abs()
    reg_anomaly_pred = (test_resid > resid_threshold).astype(int)

    prec = (
        ((reg_anomaly_pred == 1) & (spike_proxy_test == 1)).sum() /
        max((reg_anomaly_pred == 1).sum(), 1)
    )
    rec = (
        ((reg_anomaly_pred == 1) & (spike_proxy_test == 1)).sum() /
        max((spike_proxy_test == 1).sum(), 1)
    )
    f1 = 2 * prec * rec / max(prec + rec, 1e-12)

    benchmark_rows.append({
        "Model": model_name,
        "Anomaly Rate (%)": round(100 * reg_anomaly_pred.mean(), 3),
        "Spike Coverage (%)": round(100 * rec, 2),
        "Precision Proxy (%)": round(100 * prec, 2),
        "F1 Proxy": round(f1, 3)
    })

# One-Class SVM anomaly row
ocsvm_test_mask = (df.loc[X_test.index, "OCSVM_Anomaly"] == -1).astype(int)
ocsvm_prec = (
    ((ocsvm_test_mask == 1) & (spike_proxy_test == 1)).sum() /
    max((ocsvm_test_mask == 1).sum(), 1)
)
ocsvm_rec = (
    ((ocsvm_test_mask == 1) & (spike_proxy_test == 1)).sum() /
    max((spike_proxy_test == 1).sum(), 1)
)
ocsvm_f1 = 2 * ocsvm_prec * ocsvm_rec / max(ocsvm_prec + ocsvm_rec, 1e-12)

benchmark_rows.append({
    "Model": "One-Class SVM",
    "Anomaly Rate (%)": round(100 * ocsvm_test_mask.mean(), 3),
    "Spike Coverage (%)": round(100 * ocsvm_rec, 2),
    "Precision Proxy (%)": round(100 * ocsvm_prec, 2),
    "F1 Proxy": round(ocsvm_f1, 3)
})

benchmark_df = pd.DataFrame(benchmark_rows)
print("=== Anomaly-Metric Benchmark Table ===")
display(benchmark_df.sort_values("F1 Proxy", ascending=False).reset_index(drop=True))

### 2.3 Model Selection

**One-Class SVM** is the best overall for this use case.

**Why One-Class SVM wins here:**

- **Very low Anomaly Rate (~1.15%)**: flags very few points, reducing alarm fatigue in operations.
- **High Spike Coverage (~85.71%)**: catches most true spike/damage events.
- **Much higher Precision Proxy (~26.09%)**: flagged anomalies are far more likely to be real.
- **Highest F1 Proxy (~0.400)**: best balance of detection power and false-alarm control.

**Why others are weaker:**

- **Linear Regression**: very high coverage but too many false positives, so low practical reliability.
- **Random Forest**: near-zero useful spike detection in this setup.
- **Gradient Boosting / XGBoost**: better than Random Forest, but still low precision and lower F1 than One-Class SVM.

**Conclusion:** One-Class SVM is selected as the primary anomaly-detection model for downstream risk scoring and prescriptive maintenance.

---

## Part 3 — Feature Engineering

We create domain-specific engineered features to capture electrical, thermal, mechanical, and environmental behavior patterns that may indicate anomalies or degradation.

In [ ]:
import numpy as np
import pandas as pd

fe_df = df.copy()

# ---------------- ELECTRICAL ----------------
fe_df["Power_Index"] = fe_df["Voltage_V"] * fe_df["Current_Amp"]
fe_df["Current_Deviation"] = fe_df["Current_Amp"] - fe_df["Current_Amp"].rolling(window=30).mean()

# ---------------- THERMAL ----------------
fe_df["Thermal_Gradient"] = fe_df["Winding_Temp_C"] - fe_df["Bearing_Temp_C"]
fe_df["Winding_Temp_Rate"] = fe_df["Winding_Temp_C"].diff()

# ---------------- MECHANICAL ----------------
fe_df["Vibration_RPM_Ratio"] = fe_df["Vibration_mm_s"] / fe_df["Motor_RPM"]
fe_df["Vibration_Energy"] = fe_df["Vibration_mm_s"].rolling(window=20).mean()

# ---------------- COOLING / ENV ----------------
fe_df["Cooling_Efficiency"] = fe_df["Coolant_Pressure_Bar"] / fe_df["Winding_Temp_C"]
fe_df["Humidity_Thermal_Stress"] = fe_df["Ambient_Humidity_Pct"] * fe_df["Winding_Temp_C"]

fe_df = fe_df.dropna().reset_index(drop=True)

### 3.1 OCSVM-Based Feature Impact (Permutation Sensitivity)

We train One-Class SVM on engineered features and estimate feature impact by **permutation sensitivity**:
for each feature, we shuffle values and measure how much the OCSVM anomaly score changes. Larger score shift indicates stronger contribution to anomaly detection.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Numeric engineered feature set
drop_if_present = [
    "Anomaly", "OCSVM_Anomaly", "OCSVM_Score",
    "Anomaly_Severity", "Anomaly_Risk_Score"
]
X_anomaly = fe_df.select_dtypes(include=[np.number]).drop(
    columns=[c for c in drop_if_present if c in fe_df.columns],
    errors="ignore"
)

# Scale and train OCSVM on engineered features
fe_scaler = StandardScaler()
X_fe_scaled = fe_scaler.fit_transform(X_anomaly)
ocsvm_fe = OneClassSVM(kernel="rbf", nu=0.05, gamma="scale")
ocsvm_fe.fit(X_fe_scaled)

# Baseline anomaly score (higher means more anomalous)
base_scores = -ocsvm_fe.decision_function(X_fe_scaled)

# Permutation sensitivity
rng = np.random.RandomState(42)
n_repeats = 8
feature_impacts = []

for col_idx, col_name in enumerate(X_anomaly.columns):
    shifts = []
    for _ in range(n_repeats):
        X_perm = X_fe_scaled.copy()
        X_perm[:, col_idx] = rng.permutation(X_perm[:, col_idx])
        perm_scores = -ocsvm_fe.decision_function(X_perm)
        shifts.append(np.mean(np.abs(perm_scores - base_scores)))
    feature_impacts.append(np.mean(shifts))

feature_impacts = np.array(feature_impacts)
importances = feature_impacts / (feature_impacts.max() + 1e-12)
feature_names = X_anomaly.columns
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("OCSVM permutation sensitivity computed.")
print("Top 5 impactful engineered features:")
display(importance_df.head(15))

### 3.2 Plotting Feature Importance

Visualize which features have the highest predictive power for identifying anomalies.

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"], color="#5B9BD5")
plt.gca().invert_yaxis()
plt.title("Feature Impact for Anomaly Detection (One-Class SVM + Permutation)")
plt.xlabel("Normalised Permutation Impact")
plt.tight_layout()
plt.show()

### 3.3 Prescriptive Analysis

Based on engineered feature thresholds, we define rule-based prescriptive actions that recommend specific maintenance interventions.

In [ ]:
# Calculate quantile threshold once before applying function
power_index_95 = fe_df["Power_Index"].quantile(0.95)

def prescriptive_engineering_rules(row):
    actions = []

    if row["Vibration_Energy"] > 4:
        actions.append("Inspect bearings and lubrication")

    if row["Vibration_RPM_Ratio"] > 0.004:
        actions.append("Check shaft alignment")

    if row["Thermal_Gradient"] > 20:
        actions.append("Inspect insulation and heat dissipation")

    if row["Winding_Temp_Rate"] > 0.5:
        actions.append("Reduce load or enhance cooling")

    if row["Cooling_Efficiency"] < 0.03:
        actions.append("Inspect coolant flow and pressure")

    if row["Humidity_Thermal_Stress"] > 6000:
        actions.append("Improve ventilation and moisture control")

    if row["Current_Deviation"] > 10:
        actions.append("Inspect mechanical load fluctuation")

    if row["Power_Index"] > power_index_95:
        actions.append("Verify supply voltage and motor loading")

    return ", ".join(actions) if actions else "Normal operation"

fe_df["Prescriptive_Action"] = fe_df.apply(prescriptive_engineering_rules, axis=1)

print(fe_df["Prescriptive_Action"].value_counts().head(10))

---

## Part 4 — Evaluating Risk Scores

We compute a multivariate severity proxy and combine it with One-Class SVM anomaly evidence plus rule-based spike evidence to produce a final risk score.

### 4.1 Defining Severity Metrics

We standardize sensor and engineered features and compute a Euclidean severity proxy. This captures how far each row is from typical multivariate behavior.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import numpy as np

# Select relevant features
feature_cols = [
    "Voltage_V", "Current_Amp", "Motor_RPM",
    "Vibration_mm_s", "Winding_Temp_C", "Bearing_Temp_C",
    "Coolant_Pressure_Bar", "Ambient_Humidity_Pct",
    "Power_Index", "Current_Deviation", "Thermal_Gradient",
    "Winding_Temp_Rate", "Vibration_RPM_Ratio",
    "Vibration_Energy", "Cooling_Efficiency",
    "Humidity_Thermal_Stress"
]

X_risk = fe_df[feature_cols]

# Standardize and compute multivariate distance as severity proxy
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_risk)
anomaly_severity = np.linalg.norm(X_scaled, axis=1)

severity_scaler = MinMaxScaler()
fe_df["Anomaly_Severity"] = severity_scaler.fit_transform(
    anomaly_severity.reshape(-1, 1)
).ravel()

print(
    f"Anomaly_Severity range: {fe_df['Anomaly_Severity'].min():.4f} to "
    f"{fe_df['Anomaly_Severity'].max():.4f}"
)

### 4.2 Building Composite Risk Score (OCSVM + Rules + Severity)

No Random Forest is used here. We combine three signals:

1. OCSVM anomaly score (primary, unsupervised)
2. Rule-based vibration spike flag from Part 1.5
3. Multivariate severity proxy from standardized features

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

fe_len = len(fe_df)

# Align df-level signals to fe_df rows
if "Timestamp" in fe_df.columns and "Timestamp" in df.columns:
    align_df = (
        df.set_index("Timestamp")
          .reindex(fe_df["Timestamp"])
)
else:
    align_df = df.iloc[-fe_len:].copy()

# Signal 1: rule-based spike flag
rule_flag = (align_df["Vibration_mm_s"].fillna(0) > vibration_threshold).astype(float).values

# Signal 2: OCSVM anomaly score (higher means more anomalous)
if "OCSVM_Score" in align_df.columns:
    ocsvm_raw = align_df["OCSVM_Score"].fillna(0).values
else:
    ocsvm_raw = np.zeros(fe_len, dtype=float)

ocsvm_norm = MinMaxScaler().fit_transform(ocsvm_raw.reshape(-1, 1)).ravel()

# Signal 3: multivariate severity proxy from engineered features
severity_norm = fe_df["Anomaly_Severity"].fillna(0).values

# Weighted composite risk score (no RF)
composite_risk = 0.55 * ocsvm_norm + 0.30 * rule_flag + 0.15 * severity_norm
fe_df["Anomaly_Risk_Score"] = composite_risk

print(
    f"Risk score stats -> min: {composite_risk.min():.4f}, "
    f"max: {composite_risk.max():.4f}, mean: {composite_risk.mean():.4f}"
)
print(
    f"Rows with non-zero risk signal: {(composite_risk > 0).sum()} "
    f"({100 * (composite_risk > 0).mean():.2f}% of fe_df)"
)

### 4.3 Risk Score Quality Check

Review risk-score distribution and inspect top-risk rows before final level classification.

In [ ]:
q = fe_df["Anomaly_Risk_Score"].quantile([0.50, 0.75, 0.90, 0.95, 0.99])
print("Risk score quantiles:")
display(q.to_frame(name="Risk_Score_Quantile"))

top_cols = [
    "Timestamp", "Current_Amp", "Vibration_mm_s",
    "Winding_Temp_C", "Bearing_Temp_C",
    "Anomaly_Severity", "Anomaly_Risk_Score"
]
top_cols = [c for c in top_cols if c in fe_df.columns]

print("Top 10 highest-risk rows:")
display(fe_df[top_cols].sort_values("Anomaly_Risk_Score", ascending=False).head(10))

### 4.4 Classifying Risk Levels

Categorize the continuous risk score into **Low**, **Medium**, and **High** using data-driven percentile cutoffs.

In [ ]:
q75 = fe_df["Anomaly_Risk_Score"].quantile(0.75)
q90 = fe_df["Anomaly_Risk_Score"].quantile(0.90)

print(f"Risk cutoff (Low/Medium): q75 = {q75:.4f}")
print(f"Risk cutoff (Medium/High): q90 = {q90:.4f}")

def risk_level(score):
    if score < q75:
        return "Low"
    elif score < q90:
        return "Medium"
    return "High"

fe_df["Anomaly_Risk_Level"] = fe_df["Anomaly_Risk_Score"].apply(risk_level)
print("\nRisk level counts:")
print(fe_df["Anomaly_Risk_Level"].value_counts())

### 4.5 Final Results

Display the consolidated output with key sensor readings, engineered features, and risk assessments.

In [ ]:
final_cols = [
    "Timestamp",
    "Motor_ID",

    # Raw sensor values
    "Current_Amp",
    "Motor_RPM",
    "Vibration_mm_s",
    "Winding_Temp_C",
    "Bearing_Temp_C",

    # Engineered features (important ones)
    "Thermal_Gradient",
    "Vibration_Energy",
    "Cooling_Efficiency",

    # Risk outputs
    "Anomaly_Risk_Score",
    "Anomaly_Risk_Level"
]

display(fe_df[final_cols].head(10))

### 4.6 Risk Level Distribution

Visualize the distribution of anomaly risk levels across the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Count category distribution
risk_counts = fe_df["Anomaly_Risk_Level"].value_counts()

# Bar plot
plt.figure(figsize=(6, 4))
plt.bar(risk_counts.index, risk_counts.values, color=["green", "orange", "red"])
plt.title("Anomaly Risk Level Distribution")
plt.xlabel("Risk Level")
plt.ylabel("Number of Instances")
plt.tight_layout()
plt.show()

# Pie chart
plt.figure(figsize=(6, 6))
plt.pie(
    risk_counts.values,
    labels=risk_counts.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=["green", "orange", "red"]
)
plt.title("Anomaly Risk Level Percentage Distribution")
plt.tight_layout()
plt.show()